# HAETAE-IRV: 전체 서명 결함주입 실험 (STM32F4, CW-Husky)

**위에서부터 순서대로** 실행. 모든 함수는 SETUP의 한 셀에 모아 정의(커널 재시작에 강함).

- 변형: `baseline`/`double`/`leeha`/`irv`. 클럭: **HSE 직결 7.37MHz**(글리치 가능).
- 결함지점(슈도라인): UNPACK(1)·SEED(2)·SIGNBIT(3)·LSB(5)·CS=T1(7)·ADDY=T2(8)·REJECT=RB(9).
- 커맨드: `e`echo/`k`keygen/`p`서명(16B=메시지)/`t`사이클 · FAULT_SIM: `f`[라인,oneshot,checkskip]/`x`z1/`s`참s1/`c`챌린지.

**순서**: SETUP → BUILD → SMOKE → EXP1(T2직접) → EXP2(커버리지) → EXP3(재현율) → EXP4(2차결함) → EXP5(오버헤드) → EXP6(연산시간).

## SETUP

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM  = 'CW308_STM32F4'
CRYPTO_TARGET = 'NONE'
SS_VER = 'SS_VER_1_1'

In [2]:
%matplotlib inline
import chipwhisperer as cw
import numpy as np, matplotlib.pyplot as plt
import time, struct, csv, collections, statistics
scope = cw.scope(name='Husky')

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

In [4]:
%%bash
cd ../../../
ls firmware/mcu/hal/chipwhisperer-fw-extra/stm32f4/Makefile.stm32f4 >/dev/null 2>&1 && echo 'F4 HAL OK' || git submodule update --init firmware/mcu/hal/chipwhisperer-fw-extra

F4 HAL OK


In [5]:
# ===== 클럭 + 모든 헬퍼/함수 정의 (이 셀 하나로 전부) =====
scope.clock.clkgen_freq = 7.37e6
scope.clock.adc_mul = 1
scope.io.hs2 = 'clkgen'
time.sleep(0.2)
print('clkgen_freq =', scope.clock.clkgen_freq, ' hs2 =', scope.io.hs2)

import importlib, haetae_recover; importlib.reload(haetae_recover)
from haetae_recover import attack_recover

FW = '../../../firmware/mcu/simpleserial-haetae/'
FL = {'NONE':0,'SEED':1,'SIGNBIT':2,'UNPACK':3,'LSB':4,'CS':5,'ADDY':6,'REJECT':7}
FAULT_LINES = ['NONE','SEED','SIGNBIT','UNPACK','LSB','CS','ADDY','REJECT']
RESULTS = collections.OrderedDict()

def flash(hexname):
    cw.program_target(scope, prog, FW + hexname)
    reset_target(scope); time.sleep(0.5); target.flush()
def recover_target():
    reset_target(scope); time.sleep(0.5); target.flush()
def ss_echo(timeout=3000):
    target.flush(); target.simpleserial_write('e', bytearray())
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def ss_keygen(timeout=60000):
    target.flush(); target.simpleserial_write('k', bytearray())
    r = target.simpleserial_read('r', 1, timeout=timeout); return bytes(r) if r else None
def ss_sign_msg(msg16, timeout=90000):
    target.flush(); target.simpleserial_write('p', bytearray(msg16))
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def ss_sign(timeout=90000):
    return ss_sign_msg([0]*16, timeout)
def ss_cycles(timeout=120000):
    target.flush(); target.simpleserial_write('t', bytearray([0]*16))
    r = target.simpleserial_read('r', 4, timeout=timeout); return struct.unpack('<I', bytes(r))[0] if r else None
def set_fault(line, oneshot=0, checkskip=0):
    target.flush(); target.simpleserial_write('f', bytes([line, oneshot, checkskip]))
    return target.simpleserial_read('r', 1, timeout=3000)

def sweep_variant(label, recover_t2=False, timeout=90000):
    print('[%s]' % label); res = {}
    for name in FAULT_LINES:
        set_fault(FL[name], 0)
        d = ss_sign(timeout=timeout); res[name] = d.hex() if d else None
        tag = ''
        if d is None: recover_target()
        elif recover_t2 and name == 'ADDY':
            try: tag = ' | T2 s1=%.0f%%' % (100*attack_recover(target)['agreement'])
            except Exception: tag = ' | recover_err'
        print('   %-8s %s%s' % (name, res[name], tag))
    RESULTS[label] = res; return res

def measure_success(faults, M=10, reps=3, verify_t2=('ADDY',), timeout=90000):
    msgs = [[(j & 0xFF), ((j >> 8) & 0xFF)] + [0]*14 for j in range(M*reps)]
    clean = {}
    for j, msg in enumerate(msgs):
        set_fault(FL['NONE'], 0); d = ss_sign_msg(msg, timeout)
        if d is None: recover_target(); set_fault(FL['NONE'],0); d = ss_sign_msg(msg, timeout)
        clean[j] = d.hex() if d else None
    print('clean 캐시 %d/%d' % (sum(v is not None for v in clean.values()), len(msgs)))
    rows = collections.OrderedDict()
    for fn in faults:
        brate, bkey = [], []
        for b in range(reps):
            man = key = n = 0
            for i in range(M):
                j = b*M + i
                if clean[j] is None: continue
                n += 1; set_fault(FL[fn], 1); d = ss_sign_msg(msgs[j], timeout)
                if d is None: recover_target(); continue
                if d.hex() != clean[j]:
                    man += 1
                    if fn in verify_t2:
                        try:
                            r = attack_recover(target)
                            if r['clean_T2'] and r['agreement'] == 1.0: key += 1
                        except Exception: pass
            brate.append(100.0*man/max(n,1))
            if fn in verify_t2: bkey.append(100.0*key/max(n,1))
        r = statistics.mean(brate); rows[fn] = {'rate': r, 'batches': brate}
        att = ('%.1f' % (100.0/r)) if r > 0 else 'inf'
        line = '%-8s 재현율 %.1f%% %s | ~%s회/누설' % (fn, r, ['%.0f' % x for x in brate], att)
        if fn in verify_t2:
            k = statistics.mean(bkey); rows[fn]['key'] = k; line += ' | 키복원 %.1f%% of %s' % (k, ['%.0f'%x for x in bkey])
        print(line)
    return rows

print('defs OK — 모든 함수 정의 완료')

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


clkgen_freq = 7371428.571428572  hs2 = clkgen
defs OK — 모든 함수 정의 완료


## BUILD — 8개 hex 한 번에 (clean 4 + FAULT_SIM 4). ~3분

In [6]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-haetae
for V in baseline double leeha irv; do
  make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1
  make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1 && cp simpleserial-haetae-$1.hex haetae-$V-$1.hex
  make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V FAULT_SIM=1 >/dev/null 2>&1
  make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V FAULT_SIM=1 >/dev/null 2>&1 && cp simpleserial-haetae-$1.hex haetae-$V-FSIM-$1.hex
  echo "built $V (clean + FSIM)"
done

built baseline (clean + FSIM)
built double (clean + FSIM)
built leeha (clean + FSIM)
built irv (clean + FSIM)


## SMOKE — baseline 통신 + GOLDEN + 사이클

In [7]:
flash('haetae-baseline-{}.hex'.format(PLATFORM))
print('echo  :', (ss_echo() or b'').hex(), ' (a0a1..af 기대)')
print('keygen:', (ss_keygen() or b'').hex(), ' (00 기대)')
g1 = ss_sign(); g2 = ss_sign()
print('sign#1:', g1.hex() if g1 else None)
print('sign#2:', g2.hex() if g2 else None)
assert g1 and g1 == g2, '재현성 실패'
GOLDEN = g1
print('GOLDEN(F4) =', GOLDEN.hex())
print('baseline sign cycles =', ss_cycles())

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25123 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25123 bytes
echo  : a0a1a2a3a4a5a6a7a8a9aaabacadaeaf  (a0a1..af 기대)
keygen: 00  (00 기대)
sign#1: ba9f152c607b207fc6512635ba11388c
sign#2: ba9f152c607b207fc6512635ba11388c
GOLDEN(F4) = ba9f152c607b207fc6512635ba11388c
baseline sign cycles = 60858575


## EXP1 — T2 직접 추출 (baseline FSIM): 단일 서명 s1 100% 복원

In [8]:
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
set_fault(FL['NONE']); g = ss_sign(); print('FL_NONE :', g.hex() if g else None, '(=GOLDEN)')
set_fault(FL['ADDY']); d = ss_sign(); print('FL_ADDY :', d.hex() if d else None, '(T2 누설)')
res = attack_recover(target)
print('clean_T2 =', res['clean_T2'], '| s1 일치율 = %.1f%%' % (100*res['agreement']), '| sign', res['sign'])

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
FL_NONE : ba9f152c607b207fc6512635ba11388c (=GOLDEN)
FL_ADDY : 63ff5ebfaa6263739651890939cccb48 (T2 누설)
clean_T2 = True | s1 일치율 = 100.0% | sign -


## EXP2 — 커버리지 (4변형 × 7결함, 1메시지) → 표 A
`golden`=무영향 · `LEAK`=baseline과 동일(미차단) · `blocked`=무효화/난수화. baseline FL_ADDY는 EXP1에서 s1 100%로 'LEAK' 실효성 입증.

In [9]:
RESULTS.clear()
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM)); sweep_variant('baseline', recover_t2=True)
for V in ['double','leeha','irv']:
    flash('haetae-{}-FSIM-{}.hex'.format(V, PLATFORM)); print('echo:', (ss_echo() or b'').hex()[:8])
    sweep_variant(V)

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
[baseline]
   NONE     ba9f152c607b207fc6512635ba11388c
   SEED     7d68d2845d9bce2ceef3591fda4583a7
   SIGNBIT  d6c977b847eeb9f2c30ac1ab4a8fa4c5
   UNPACK   99f2b8e52e112a5863677bdfde153837
   LSB      4f6bb1ef6098c2c9abbba07930f73bb3
   CS       9b46b88b2dfcbf53169ed3a778024414
   ADDY     63ff5ebfaa6263739651890939cccb48 | T2 s1=100%
   REJECT   ab4a71a7f8084e289653c64b8105a54d
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25891 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25891 bytes
echo: a0a1a2a3
[double]
   NONE     ba9f152c607b207fc6512635ba11388c
   SEED     a70948f5783c58a8596fa54e76dd1e9a
   SIGNBIT  a70948f5783c58a8596fa54e76dd1e9a
 

In [10]:
GH = GOLDEN.hex(); base = RESULTS['baseline']
def classify(label, name):
    d = RESULTS[label].get(name)
    if d is None:           return 'MUTE'
    if d == GH:             return 'golden'
    if d == base.get(name): return 'LEAK'
    return 'blocked'
cols = list(RESULTS.keys())
hdr = '%-9s | ' % 'fault' + ' | '.join('%-8s' % c for c in cols); print(hdr); print('-'*len(hdr))
for name in FAULT_LINES:
    if name == 'NONE': continue
    print('%-9s | ' % name + ' | '.join('%-8s' % classify(c, name) for c in cols))
print('\nFL_NONE 오탐:', {c: ('OK' if RESULTS[c].get('NONE')==GH else 'FAIL') for c in cols})
with open('haetae_f4_coverage.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['fault']+cols)
    for name in FAULT_LINES:
        if name=='NONE': continue
        w.writerow([name]+[classify(c,name) for c in cols])
print('saved haetae_f4_coverage.csv')

fault     | baseline | double   | leeha    | irv     
-----------------------------------------------------
SEED      | LEAK     | blocked  | blocked  | blocked 
SIGNBIT   | LEAK     | blocked  | blocked  | blocked 
UNPACK    | LEAK     | blocked  | blocked  | blocked 
LSB       | LEAK     | blocked  | blocked  | blocked 
CS        | LEAK     | blocked  | blocked  | blocked 
ADDY      | LEAK     | blocked  | blocked  | blocked 
REJECT    | LEAK     | blocked  | LEAK     | blocked 

FL_NONE 오탐: {'baseline': 'OK', 'double': 'OK', 'leeha': 'OK', 'irv': 'OK'}
saved haetae_f4_coverage.csv


## EXP3 — 누설 재현율 (baseline, one-shot, 메시지 가변). 빠르게 보려면 M=5

In [11]:
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
SR = measure_success(['SEED','SIGNBIT','UNPACK','LSB','CS','ADDY','REJECT'], M=10, reps=3)
with open('haetae_f4_success_rate.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['fault','reproduce_%','attempts_per_leak','key_recover_%','batches'])
    for fn,v in SR.items():
        att=(100.0/v['rate']) if v['rate']>0 else None
        w.writerow([fn,'%.1f'%v['rate'],('%.1f'%att) if att else 'inf',('%.1f'%v['key']) if 'key' in v else '',v['batches']])
print('saved haetae_f4_success_rate.csv')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
clean 캐시 30/30
SEED     재현율 100.0% ['100', '100', '100'] | ~1.0회/누설
SIGNBIT  재현율 23.3% ['10', '10', '50'] | ~4.3회/누설
UNPACK   재현율 100.0% ['100', '100', '100'] | ~1.0회/누설
LSB      재현율 23.3% ['20', '20', '30'] | ~4.3회/누설
CS       재현율 23.3% ['10', '30', '30'] | ~4.3회/누설
ADDY     재현율 83.3% ['70', '90', '90'] | ~1.2회/누설 | 키복원 83.3% of ['70', '90', '90']
REJECT   재현율 90.0% ['90', '90', '90'] | ~1.1회/누설
saved haetae_f4_success_rate.csv


## EXP4 — 2차 결함 (T2 + 검사분기 스킵) ★ IRV 무분기 우위
detect-and-abort(double 비교·leeha 검증)는 검사분기 스킵으로 우회(LEAK), IRV는 응답연산 핵심을 무분기 감염으로 처리해 차단 유지.

In [12]:
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
set_fault(FL['ADDY'], 0, 0); LEAK2 = ss_sign(); ref = LEAK2.hex() if LEAK2 else None
print('기준 T2 누설:', ref)
print('\n=== 2차결함 (T2 ADDY + 검사분기 스킵) ===')
SO = collections.OrderedDict()
for V in ['baseline','double','leeha','irv']:
    flash('haetae-{}-FSIM-{}.hex'.format(V, PLATFORM))
    set_fault(FL['ADDY'], 0, 1); d = ss_sign()
    if d is None: recover_target()
    h = d.hex() if d else None
    verdict = 'LEAK(우회)' if h == ref else ('blocked' if h else 'MUTE')
    SO[V] = (h, verdict); print('  %-9s %s -> %s' % (V, h, verdict))
print('\n해석: double/leeha=검사분기 스킵으로 우회(LEAK), IRV=무분기라 차단 유지.')
with open('haetae_f4_2ndorder.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['variant','digest','verdict']); [w.writerow([V,h,v]) for V,(h,v) in SO.items()]
print('saved haetae_f4_2ndorder.csv')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
기준 T2 누설: 63ff5ebfaa6263739651890939cccb48

=== 2차결함 (T2 ADDY + 검사분기 스킵) ===
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25683 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25683 bytes
  baseline  63ff5ebfaa6263739651890939cccb48 -> LEAK(우회)
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25891 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25891 bytes
  double    63ff5ebfaa6263739651890939cccb48 -> LEAK(우회)
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31731

## EXP5 — 구현 오버헤드 (코드/RAM, 무결함 빌드) → 표 C

In [13]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-haetae
printf '%-10s %8s %6s %7s %8s\n' variant text data bss dec
for V in baseline double leeha irv; do
  make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1
  make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=$V >/dev/null 2>&1
  S=$(arm-none-eabi-size simpleserial-haetae-$1.elf | tail -1)
  printf '%-10s %8s %6s %7s %8s\n' $V $(echo $S | awk '{print $1, $2, $3, $4}')
done

variant        text   data     bss      dec
baseline      24556    564    6048    31168
double        24732    564    6048    31344
leeha         30788    564    6056    37408
irv           31004    564    6056    37624


## EXP6 — 연산시간 (서명 1회 DWT 사이클, 무결함 빌드) → 표 C

In [14]:
CYC = collections.OrderedDict()
for V in ['baseline','double','leeha','irv']:
    flash('haetae-{}-{}.hex'.format(V, PLATFORM))
    ss_keygen()
    CYC[V] = ss_cycles(); print('%-9s %s cycles' % (V, CYC[V]))
b = CYC.get('baseline') or 1
print('\n%-9s %-12s %s' % ('변형','사이클','상대'))
for V, c in CYC.items(): print('%-9s %-12s %.3f x' % (V, c, (c/b) if c else float('nan')))
with open('haetae_f4_timing.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['variant','sign_cycles','relative'])
    for V,c in CYC.items(): w.writerow([V, c, ('%.3f'%(c/b)) if c else ''])
print('saved haetae_f4_timing.csv')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25123 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25123 bytes
baseline  60858575 cycles
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25299 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25299 bytes
double    121717096 cycles
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31355 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31355 bytes
leeha     115851925 cycles
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31571 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31571 bytes
irv     

## 산출 CSV (→ `D:/06_github_desktop/fia_cm_haetae/test/2026-06-30/`)
- `haetae_f4_coverage.csv` (표 A) · `haetae_f4_success_rate.csv` (재현율) · `haetae_f4_2ndorder.csv` (2차결함) · `haetae_f4_timing.csv` + EXP5 size (표 C)

In [16]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-haetae
make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha >/dev/null 2>&1
make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha 2>&1 | tail -3 && cp simpleserial-haetae-$1.hex haetae-leeha-$1.hex
make clean PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha FAULT_SIM=1 >/dev/null 2>&1
make       PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 VARIANT=leeha FAULT_SIM=1 >/dev/null 2>&1 && cp simpleserial-haetae-$1.hex haetae-leeha-FSIM-$1.hex
echo 'rebuilt leeha (수정판)'
arm-none-eabi-size simpleserial-haetae-$1.elf

+ CRYPTO_TARGET = NONE
+ CRYPTO_OPTIONS = 
+--------------------------------------------------------
rebuilt leeha (수정판)
   text	   data	    bss	    dec	    hex	filename
  31156	    564	  14264	  45984	   b3a0	simpleserial-haetae-CW308_STM32F4.elf


In [17]:
flash('haetae-leeha-FSIM-{}.hex'.format(PLATFORM)); sweep_variant('leeha')
flash('haetae-leeha-{}.hex'.format(PLATFORM)); ss_keygen()
c = ss_cycles(); print('leeha cycles =', c, ' 상대 =', round(c/60858575, 3), 'x')

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31723 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31723 bytes
[leeha]
   NONE     ba9f152c607b207fc6512635ba11388c
   SEED     a70948f5783c58a8596fa54e76dd1e9a
   SIGNBIT  a70948f5783c58a8596fa54e76dd1e9a
   UNPACK   a70948f5783c58a8596fa54e76dd1e9a
   LSB      a70948f5783c58a8596fa54e76dd1e9a
   CS       a70948f5783c58a8596fa54e76dd1e9a
   ADDY     a70948f5783c58a8596fa54e76dd1e9a
   REJECT   ab4a71a7f8084e289653c64b8105a54d
Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 31315 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 31315 bytes
leeha cycles = 69175415  상대 = 1.137 x
